In [0]:
from pyspark.sql import functions as F

customers = spark.table("brazilian_ecommerce.prod.bronze_customers")

total_rows = customers.count()

def check_uniqueness(df, column: str, total: int) -> None:
    distinct_count = df.select(column).distinct().count()
    null_count = df.filter(F.col(column).isNull()).count()
    is_unique = distinct_count == total

    print(f"Column: {column}")
    print(f"  Total rows:      {total}")
    print(f"  Distinct values: {distinct_count}")
    print(f"  Nulls:           {null_count}")
    print(f"  Unique?          {is_unique}")
    print("-" * 40)

for col in ["customer_id", "customer_unique_id"]:
    check_uniqueness(customers, col, total_rows)

# show actual duplicate values (if any) for customer_unique_id
duplicates = (
    customers.groupBy("customer_unique_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)
duplicates.show(10, truncate=False)